# 04 · CNN Architectures `[local]` (benchmark section also runs on `[kaggle]`)

**Goal:** know the famous CNN architectures by the *problem each one solved*, build their key blocks yourself, and compare real pretrained models on accuracy, size, compute and **measured speed**. Plus Grad-CAM, to see where a model looks.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | The architecture timeline as a chain of problems and fixes |
| 📐 Math | Stacked 3×3s, 1×1 bottlenecks, residual gradients, inverted residuals, squeeze-excite, compound scaling |
| 🔨 From scratch | ResNet basic + bottleneck blocks, MobileNetV2 inverted residual, SE block, gradient flow with and without skips |
| 🚀 timm | 6 pretrained models scored on Imagenette with no training; params, FLOPs, latency, throughput |
| 🔎 Grad-CAM | From scratch with hooks: heatmaps of where the model looked |
| 📊 Evaluate | Accuracy vs. latency trade-off, FLOPs vs. real speed, per-class results, explained mistakes |
| 🧪 Experiment | Plain vs. residual training, input resolution, batch size vs. throughput |

**Kaggle:** run this notebook there too, to get NVIDIA GPU numbers for the benchmark table (section 4.3). Results are saved per platform, so you can compare.

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import json
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import DataLoader
from torch.utils.flop_counter import FlopCounterMode
from torchvision.datasets import ImageFolder
from torchvision import transforms as T
from cvlib.datasets import imagenette, IMAGENETTE_CLASSES, IMAGENETTE_TO_IMAGENET, cifar10
from cvlib.loaders import GPULoader, random_crop_flip, CIFAR_MEAN, CIFAR_STD
from cvlib import train as cvtrain

device = env.device
rng = np.random.default_rng(0)

def sync():
    """Wait for the GPU to finish (needed for honest timing, see 00_setup)."""
    if device == "cuda": torch.cuda.synchronize()
    elif device == "mps": torch.mps.synchronize()

---
# 🧠 1 · Intuition: each architecture fixed a problem

| Year | Model | Problem | Key idea |
|---|---|---|---|
| 1998 | **LeNet** | read digits on checks | conv → pool → conv → pool → dense: the template for everything after |
| 2012 | **AlexNet** | can deep nets work on 1M+ real photos? | GPUs, ReLU, dropout, data augmentation. Won ImageNet by a landslide and started the deep learning era |
| 2014 | **VGG** | how to go deeper simply? | only 3×3 convs, stacked. Two 3×3 = one 5×5's reach with fewer weights and an extra ReLU |
| 2014 | **Inception** | which filter size is best? | all of them, in parallel branches, with cheap 1×1 convs to shrink channels first |
| 2015 | **ResNet** | deeper nets got *worse*, even on training data | **skip connections**: each block learns a *correction* to its input. Made 50–150+ layers trainable |
| 2016 | **DenseNet** | reuse features | each layer receives all previous layers' outputs (concatenated) |
| 2017–18 | **MobileNet v1/v2** | run on phones | depthwise-separable convs; **inverted residual** blocks |
| 2017 | **SENet** | let the network focus on useful channels | **squeeze-and-excitation**: re-weight channels using global context |
| 2019 | **EfficientNet** | how to scale up a network well? | scale depth, width and resolution **together** (compound scaling) |
| 2020 | **RegNet** | design by searching for rules, not single models | simple rules for how widths should grow across stages |
| 2022 | **ConvNeXt** | can a CNN match Vision Transformers? | modernize ResNet with transformer-era choices: big 7×7 depthwise kernels, LayerNorm, GELU, fewer activations |

**The big lesson of ResNet:** in principle, a deeper network can always copy a shallower one (extra layers just pass their input through). In practice, plain deep networks couldn't even *learn* to pass inputs through. Skip connections make "do nothing" the default, so each block only has to learn what to *add*.

---
# 📐 2 · The math

### Why stack 3×3 convs (VGG)
Two 3×3 layers see a 5×5 region; three see 7×7. With $C$ channels in and out:

$$\text{one } 7{\times}7: 49C^2 \qquad \text{three } 3{\times}3: 27C^2 \;\; (45\% \text{ fewer, and 3 non-linearities instead of 1})$$

### 1×1 bottlenecks (Inception, ResNet-50)
Reduce channels with a 1×1 conv before the expensive 3×3, then expand again. With 256 channels:

$$3{\times}3 \text{ on } 256: \; 9 \cdot 256^2 = 590k \qquad \underbrace{256{\to}64}_{1{\times}1} + \underbrace{64{\to}64}_{3{\times}3} + \underbrace{64{\to}256}_{1{\times}1} = 16k + 37k + 16k = 70k$$

### Residual connection
A block computes $y = x + F(x)$. During backprop:

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y}\left(1 + \frac{\partial F}{\partial x}\right)$$

The **"1"** is a gradient highway: even if $\partial F/\partial x$ is tiny, gradient reaches early layers unchanged. Without it, gradients get multiplied by many small factors and vanish.

When the block changes shape (stride 2 or more channels), the shortcut becomes a 1×1 conv with stride 2 so $x$ and $F(x)$ can be added.

### Inverted residual (MobileNetV2)
ResNet's bottleneck goes **wide → narrow → wide**. MobileNetV2 inverts it: **narrow → wide → narrow**, with the cheap depthwise 3×3 in the wide middle:

$$\underbrace{C \to tC}_{1{\times}1 \text{ expand}} \;\to\; \underbrace{tC \to tC}_{3{\times}3 \text{ depthwise}} \;\to\; \underbrace{tC \to C}_{1{\times}1 \text{ project, no ReLU}} \quad (t = 6)$$

The skip connects the *narrow* ends, which keeps memory traffic low on phones.

### Squeeze-and-excitation
1. **Squeeze:** global-average-pool each channel → one number per channel
2. **Excite:** tiny MLP → a weight in (0, 1) per channel (sigmoid)
3. **Scale:** multiply each channel by its weight

Cheap channel attention: "given the whole image, which feature detectors matter right now?" (Attention in general comes in module 08.)

### Compound scaling (EfficientNet)
Scale depth $d$, width $w$ and input resolution $r$ together with one knob $\phi$:

$$d = \alpha^\phi,\quad w = \beta^\phi,\quad r = \gamma^\phi \qquad \text{with } \alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$$

FLOPs grow with $d \cdot w^2 \cdot r^2$, so each step of $\phi$ roughly **doubles** the compute. B0 → B7 = $\phi$ from 0 to 7.

---
# 🔨 3 · Build the key blocks yourself

In [ ]:
def n_params(m):
    return sum(p.numel() for p in m.parameters())

def macs(m, shape):
    with FlopCounterMode(display=False) as fc:
        m.eval()(torch.zeros(1, *shape))
    return fc.get_total_flops() / 2

class BasicBlock(nn.Module):
    """ResNet-18/34 block: two 3×3 convs + skip."""
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False); self.bn2 = nn.BatchNorm2d(c_out)
        self.shortcut = nn.Identity()
        if stride != 1 or c_in != c_out:                           # shapes differ → project the shortcut
            self.shortcut = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x))                      # the "+ x" is the whole idea

class Bottleneck(nn.Module):
    """ResNet-50/101/152 block: 1×1 reduce → 3×3 → 1×1 expand (×4) + skip."""
    def __init__(self, c_in, width, stride=1, expansion=4):
        super().__init__()
        c_out = width * expansion
        self.body = nn.Sequential(
            nn.Conv2d(c_in, width, 1, bias=False), nn.BatchNorm2d(width), nn.ReLU(inplace=True),
            nn.Conv2d(width, width, 3, stride, 1, bias=False), nn.BatchNorm2d(width), nn.ReLU(inplace=True),
            nn.Conv2d(width, c_out, 1, bias=False), nn.BatchNorm2d(c_out))
        self.shortcut = nn.Identity() if stride == 1 and c_in == c_out else nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        return F.relu(self.body(x) + self.shortcut(x))

class SqueezeExcite(nn.Module):
    def __init__(self, c, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(c, c // reduction), nn.ReLU(inplace=True), nn.Linear(c // reduction, c), nn.Sigmoid())

    def forward(self, x):
        w = self.fc(x.mean(dim=(2, 3)))                            # squeeze: (B, C) ; excite: weights in (0, 1)
        return x * w[:, :, None, None]                             # scale every channel

class InvertedResidual(nn.Module):
    """MobileNetV2 block (+ optional SE, as in MobileNetV3 / EfficientNet)."""
    def __init__(self, c_in, c_out, stride=1, t=6, se=False):
        super().__init__()
        mid = c_in * t
        layers = [nn.Conv2d(c_in, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.ReLU6(inplace=True),               # expand
                  nn.Conv2d(mid, mid, 3, stride, 1, groups=mid, bias=False), nn.BatchNorm2d(mid), nn.ReLU6(inplace=True)]  # depthwise
        if se:
            layers.append(SqueezeExcite(mid))
        layers += [nn.Conv2d(mid, c_out, 1, bias=False), nn.BatchNorm2d(c_out)]                                  # project (linear!)
        self.body = nn.Sequential(*layers)
        self.use_skip = stride == 1 and c_in == c_out

    def forward(self, x):
        out = self.body(x)
        return x + out if self.use_skip else out

blocks = {
    "plain: two 3×3, 256→256": nn.Sequential(nn.Conv2d(256, 256, 3, 1, 1, bias=False), nn.BatchNorm2d(256), nn.ReLU(), nn.Conv2d(256, 256, 3, 1, 1, bias=False), nn.BatchNorm2d(256)),
    "ResNet basic, 256→256": BasicBlock(256, 256),
    "ResNet bottleneck, 256→256": Bottleneck(256, 64),
    "MobileNetV2, 64→64 (t=6)": InvertedResidual(64, 64),
    "MobileNetV2 + SE, 64→64": InvertedResidual(64, 64, se=True),
}
print(f"{'block':<30}{'params':>10}{'MACs @ 28×28':>16}")
for name, b in blocks.items():
    c = 256 if "256" in name else 64
    print(f"{name:<30}{n_params(b):>10,}{macs(b, (c, 28, 28)):>16,.0f}")

The bottleneck processes 256-channel features with **~1/17 the parameters and compute** of two plain 3×3 convs (1/8 of a single 3×3, as computed in section 2). That's how ResNet-50 is much deeper than ResNet-34 at a similar cost. Note the skip connection itself adds **no parameters** (when shapes match).

### 3.2 · The gradient highway, measured

Build a 30-block network with and without skips, push one batch through, backprop, and measure the gradient that reaches the **first** layer. No training needed, just one backward pass at initialization.

In [ ]:
class Stack(nn.Module):
    def __init__(self, depth=30, c=32, skip=True):
        super().__init__()
        self.stem = nn.Conv2d(3, c, 3, 1, 1)
        self.blocks = nn.ModuleList([nn.Sequential(nn.Conv2d(c, c, 3, 1, 1), nn.ReLU()) for _ in range(depth)])
        self.skip = skip
        self.head = nn.Linear(c, 10)

    def forward(self, x):
        x = self.stem(x)
        for b in self.blocks:
            x = x + b(x) * 0.5 if self.skip else b(x)              # residual vs plain (0.5 keeps the residual sum from growing)
        return self.head(x.mean((2, 3)))

x_g, y_g = torch.randn(16, 3, 32, 32), torch.randint(0, 10, (16,))
for skip in (False, True):
    torch.manual_seed(0)
    net = Stack(skip=skip)
    F.cross_entropy(net(x_g), y_g).backward()
    first = net.stem.weight.grad.norm().item()
    last = net.blocks[-1][0].weight.grad.norm().item()
    print(f"{'residual' if skip else 'plain':>8}: gradient norm at first layer {first:.2e} | at last block {last:.2e} | ratio {first / last:.1e}")

In the plain network the gradient shrinks by orders of magnitude on its way to the first layer, so early layers barely learn. With skips it arrives at full strength. (Experiment 1 shows what this means for actual training.)

---
# 🚀 4 · Pretrained models with `timm`

[`timm`](https://github.com/huggingface/pytorch-image-models) (PyTorch Image Models) has 1,000+ architectures with pretrained weights behind one function: `timm.create_model(name, pretrained=True)`. It's the standard model zoo for image backbones.

In [ ]:
print(f"{len(timm.list_models(pretrained=True)):,} pretrained models available")
print("some ResNets:", timm.list_models("resnet5*", pretrained=True)[:6])

r50 = timm.create_model("resnet50.a1_in1k", pretrained=True).eval()
cfg = timm.data.resolve_data_config({}, model=r50)
print("\nresnet50 data config:", {k: cfg[k] for k in ("input_size", "interpolation", "mean", "std", "crop_pct")})

**Always use the model's own data config** (input size, resize method, mean/std normalization). Feeding a model images preprocessed differently from training is a silent accuracy killer.

### 4.1 · ResNet-50's first-layer filters

Big 7×7 filters trained on ImageNet: now you can clearly see the oriented edge detectors and color-contrast detectors that module 03's tiny 3×3 filters build up over several layers.

In [ ]:
w = r50.conv1.weight.detach()                                   # (64, 3, 7, 7)
w = (w - w.min()) / (w.max() - w.min())
fig, axes = plt.subplots(4, 16, figsize=(13, 3.6))
for ax, f in zip(axes.flat, w):
    ax.imshow(f.permute(1, 2, 0)); ax.axis("off")
plt.suptitle("ResNet-50 conv1: 64 filters, 7×7×RGB"); plt.tight_layout(); plt.show()

### 4.2 · Scoring pretrained models on Imagenette, with no training

**Imagenette** = 10 easy classes taken from ImageNet. Since the classes are real ImageNet classes, an ImageNet-trained model can be scored directly: run it, keep only the 10 relevant outputs (out of 1,000), take the largest. Zero training. A clean way to compare architectures on equal terms.

In [ ]:
root = imagenette(160)
print("classes:", IMAGENETTE_CLASSES)
print("ImageNet indices:", IMAGENETTE_TO_IMAGENET)
print("val images:", len(ImageFolder(root / "val")))

MODELS = {
    "ResNet-18": "resnet18.a1_in1k",
    "ResNet-50": "resnet50.a1_in1k",
    "MobileNetV3-L": "mobilenetv3_large_100.ra_in1k",
    "EfficientNet-B0": "efficientnet_b0.ra_in1k",
    "RegNetY-800MF": "regnety_008.pycls_in1k",
    "ConvNeXt-T": "convnext_tiny.fb_in1k",
}
subset = torch.tensor(IMAGENETTE_TO_IMAGENET)

def evaluate_imagenette(model, name):
    cfg = timm.data.resolve_data_config({}, model=model)
    ds = ImageFolder(root / "val", transform=timm.data.create_transform(**cfg))
    dl = DataLoader(ds, batch_size=64, num_workers=0)
    preds, labels = [], []
    model = model.eval().to(device)
    with torch.no_grad():
        for x, y in dl:
            preds.append(model(x.to(device))[:, subset.to(device)].argmax(1).cpu()); labels.append(y)
    return torch.cat(preds), torch.cat(labels), cfg

results = {}
for nice, tname in MODELS.items():
    t0 = time.perf_counter()
    m = timm.create_model(tname, pretrained=True)
    p, y, cfg_m = evaluate_imagenette(m, nice)
    results[nice] = dict(acc=(p == y).float().mean().item(), preds=p, labels=y, input=cfg_m["input_size"][1:], params=n_params(m) / 1e6)
    print(f"{nice:<16} acc {results[nice]['acc']:.4f} | input {cfg_m['input_size'][1]}px | {time.perf_counter() - t0:.0f}s")
    m.cpu(); del m
    if device == "mps": torch.mps.empty_cache()

### 4.3 · Size, compute and measured speed

- **Params**: memory and download size
- **GFLOPs**: compute per image, at each model's own input size
- **Latency** (batch 1): time for a single image, what an interactive app feels
- **Throughput** (batch 32): images per second when batching, what an offline pipeline cares about

Timing uses warm-up runs and GPU synchronization (see `00_setup`). Module 15 goes much deeper.

In [ ]:
def bench(model, size, batch, reps=10):
    x = torch.randn(batch, 3, size, size, device=device)
    with torch.no_grad():
        for _ in range(3): model(x)                                  # warm-up
        sync(); t0 = time.perf_counter()
        for _ in range(reps): model(x)
        sync()
    return (time.perf_counter() - t0) / reps

for nice, tname in MODELS.items():
    m = timm.create_model(tname, pretrained=False).eval()
    size = results[nice]["input"][0]
    with FlopCounterMode(display=False) as fc, torch.no_grad():
        m(torch.zeros(1, 3, size, size))
    m = m.to(device)
    results[nice]["gflops"] = fc.get_total_flops() / 1e9
    results[nice]["latency_ms"] = bench(m, size, 1, reps=20) * 1000
    results[nice]["img_per_s"] = 32 / bench(m, size, 32, reps=5)
    m.cpu(); del m
    if device == "mps": torch.mps.empty_cache()

print(f"device: {device}\n{'model':<16}{'acc':>7}{'params(M)':>11}{'GFLOPs':>8}{'latency(ms)':>13}{'img/s @32':>11}")
for k, r in results.items():
    print(f"{k:<16}{r['acc']:>7.3f}{r['params']:>11.1f}{r['gflops']:>8.2f}{r['latency_ms']:>13.1f}{r['img_per_s']:>11.0f}")

out_file = env.out_dir / f"arch_benchmark_{env.platform}_{device}.json"
json.dump({k: {kk: vv for kk, vv in r.items() if kk not in ("preds", "labels")} for k, r in results.items()}, open(out_file, "w"), indent=1, default=float)
print(f"\nsaved {out_file.name}: run this notebook on Kaggle too and compare the GPU numbers")

---
# 🔎 5 · Grad-CAM from scratch: where did the model look?

**Grad-CAM** answers "which regions made the model say *X*?":
1. Take the last conv stage's feature maps $A_k$ (e.g. 2048 maps of 7×7 for ResNet-50).
2. Backprop the class score $y^c$ to those maps; average each map's gradient → **importance weight** $w_k = \text{mean}_{i,j}\, \partial y^c / \partial A_k^{ij}$
3. Heatmap $= \text{ReLU}\left(\sum_k w_k A_k\right)$, upsampled to the image size.

We grab the feature maps and their gradients with PyTorch **hooks**: functions that run whenever a layer runs forward.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model = model.eval()
        layer.register_forward_hook(self._save)

    def _save(self, module, inp, out):
        self.acts = out
        if out.requires_grad:                                          # skip under torch.no_grad() (plain inference)
            out.register_hook(lambda g: setattr(self, "grads", g))  # called during backward

    def __call__(self, x, class_idx=None):
        self.model.zero_grad()
        logits = self.model(x)
        if class_idx is None:
            class_idx = int(logits[0, subset].argmax())                # restrict to the 10 Imagenette classes
            class_idx = int(subset[class_idx])
        logits[0, class_idx].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)                  # importance of each feature map
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu(), class_idx

r50 = r50.to(device)
cam_model = GradCAM(r50, r50.layer4)                                 # last conv stage
tf = timm.data.create_transform(**cfg)
show_tf = T.Compose(tf.transforms[:-2])                              # same resize/crop, without ToTensor/normalize
val_ds = ImageFolder(root / "val")

def show_cams(indices, title):
    fig, axes = plt.subplots(2, len(indices), figsize=(2.6 * len(indices), 5.4))
    for col, i in enumerate(indices):
        img, label = val_ds[i]
        cam, cls = cam_model(tf(img)[None].to(device))
        pred = IMAGENETTE_TO_IMAGENET.index(cls)
        axes[0, col].imshow(show_tf(img)); axes[0, col].axis("off")
        axes[0, col].set_title(f"true: {IMAGENETTE_CLASSES[label]}", fontsize=8)
        axes[1, col].imshow(show_tf(img)); axes[1, col].imshow(cam, cmap="jet", alpha=0.45); axes[1, col].axis("off")
        axes[1, col].set_title(f"pred: {IMAGENETTE_CLASSES[pred]}", fontsize=8, color="green" if pred == label else "red")
    plt.suptitle(title); plt.tight_layout(); plt.show()

per_class_first = [int(np.nonzero(np.array(val_ds.targets) == c)[0][5]) for c in range(0, 10, 2)] + [int(np.nonzero(np.array(val_ds.targets) == c)[0][7]) for c in range(1, 10, 2)]
show_cams(per_class_first[:5], "Grad-CAM, ResNet-50: where it looked")
show_cams(per_class_first[5:], "Grad-CAM, ResNet-50: where it looked")

Mostly on the object itself: the fish, the dog, the church, the parachute canopy. When the heat sits on the *background* instead (water for "tench", sky for "parachute"), the model may be using context shortcuts. Those are the predictions to distrust when the context changes.

---
# 📊 6 · Evaluate

### 6.1 · Accuracy vs. speed: the trade-off plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for k, r in results.items():
    axes[0].scatter(r["latency_ms"], r["acc"], s=r["params"] * 12, alpha=0.6)
    axes[0].annotate(k, (r["latency_ms"], r["acc"]), fontsize=8, xytext=(5, 3), textcoords="offset points")
    axes[1].scatter(r["gflops"], r["latency_ms"], s=60)
    axes[1].annotate(k, (r["gflops"], r["latency_ms"]), fontsize=8, xytext=(5, 3), textcoords="offset points")
axes[0].set_xlabel(f"latency, batch 1 (ms, {device})"); axes[0].set_ylabel("Imagenette accuracy"); axes[0].set_title("Accuracy vs. latency (bubble = params)")
axes[1].set_xlabel("GFLOPs per image"); axes[1].set_ylabel("latency (ms)"); axes[1].set_title("Do FLOPs predict speed?")
for a in axes: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

names_ = list(results)
fl = np.array([results[k]["gflops"] for k in names_]); lat = np.array([results[k]["latency_ms"] for k in names_])
rank = lambda v: np.argsort(np.argsort(v))
print(f"rank correlation between FLOPs and latency: {np.corrcoef(rank(fl), rank(lat))[0, 1]:.2f} (1.0 = FLOPs perfectly predict the speed order)")

**How to read this:**
- On a 10-class subset of ImageNet, every modern model is accurate, so the differences are small. On the full 1,000 classes the gaps are much bigger.
- **FLOPs do not reliably predict speed.** Depthwise convolutions, squeeze-excite and many small layers have few FLOPs but use the hardware poorly, especially at batch size 1, where launching hundreds of tiny GPU operations dominates. Always **measure on your target hardware**. The ranking on Kaggle's NVIDIA GPU may differ from your Mac's.

### 6.2 · Per-class accuracy and mistakes, with Grad-CAM

In [ ]:
best = max(results, key=lambda k: results[k]["acc"])
print(f"best model: {best}")
print(f"{'class':<18}" + "".join(f"{k[:9]:>10}" for k in results))
for c, cname in enumerate(IMAGENETTE_CLASSES):
    row = [((results[k]["preds"] == c) & (results[k]["labels"] == c)).sum().item() / (results[k]["labels"] == c).sum().item() for k in results]
    print(f"{cname:<18}" + "".join(f"{v:>10.3f}" for v in row))

r50_res = results["ResNet-50"]
wrong_r50 = torch.nonzero(r50_res["preds"] != r50_res["labels"]).ravel()[:5].tolist()
print(f"\nResNet-50 made {len(torch.nonzero(r50_res['preds'] != r50_res['labels']))} mistakes. A few, explained by Grad-CAM:")
show_cams(wrong_r50, "ResNet-50 mistakes: what was it looking at?")

Grad-CAM separates two kinds of mistakes:
- **Looked at the right thing, drew the wrong conclusion:** a look-alike, an unusual viewpoint, a tiny object.
- **Looked somewhere else entirely:** look for photos of a dog or a cassette player predicted as **"tench"** with the heat on people's **feet, legs or hands**. ImageNet's tench photos mostly show anglers holding their catch, so the model learned "human body parts → tench". That's a **shortcut**: a correlation in the training data, not the actual object.

Shortcuts can't be seen in an accuracy number. They show up only when you look at *why* the model made its mistakes, and they cause failures once the deployment data stops sharing the training data's quirks.

---
# 🧪 7 · Experiments: predict first, then run

**Experiment 1: plain vs. residual, actually training.** Two 20-layer CIFAR networks, identical except for skip connections. Predict the gap after 4 epochs.

In [ ]:
# Prediction: ...
class CifarNet(nn.Module):
    """ResNet-20 style: 3 stages × 3 blocks × 2 convs + stem + head = 20 layers."""
    def __init__(self, skip=True, width=(16, 32, 64)):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, width[0], 3, 1, 1, bias=False), nn.BatchNorm2d(width[0]), nn.ReLU())
        layers, c = [], width[0]
        for i, w_ in enumerate(width):
            for j in range(3):
                stride = 2 if (j == 0 and i > 0) else 1
                blk = BasicBlock(c, w_, stride)
                if not skip:
                    blk.shortcut = None
                layers.append(blk); c = w_
        self.layers = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 10))

    def forward(self, x):
        return self.head(self.layers(self.stem(x)))

def _plain_forward(self, x):                                         # BasicBlock without the "+ shortcut"
    return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x))))))

Xc, yc = cifar10("train"); Xc_test, yc_test = cifar10("test")
tl = GPULoader(Xc[:20_000], yc[:20_000], batch_size=128, shuffle=True, augment=random_crop_flip, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)
vl = GPULoader(Xc_test[:5000], yc_test[:5000], batch_size=1000, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)
for skip in (False, True):
    torch.manual_seed(0)
    net = CifarNet(skip=skip)
    if not skip:
        for b in net.layers:
            b.forward = _plain_forward.__get__(b)
    h = cvtrain.fit(net, tl, vl, nn.CrossEntropyLoss(), torch.optim.SGD(net.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4), epochs=4, device=device, verbose=False)
    print(f"{'ResNet-20' if skip else 'Plain-20 '}: train loss per epoch {[round(v, 3) for v in h['train_loss']]} | val acc {h['val_acc'][-1]:.3f}")

Even with BatchNorm in both, the residual version trains faster and further: lower **training** loss, not just better validation. That's the "degradation problem" ResNet solved, and the gap grows with depth (try `width` stages with more blocks).

**Experiment 2: input resolution.** ResNet-50 expects 224 px. Predict accuracy on Imagenette at 96, 160, 224 and 288 px.

In [ ]:
# Prediction: ...
for size in (96, 160, 224, 288):
    tf_s = T.Compose([T.Resize(size, interpolation=T.InterpolationMode.BICUBIC), T.CenterCrop(size), T.ToTensor(), T.Normalize(cfg["mean"], cfg["std"])])
    dl = DataLoader(ImageFolder(root / "val", transform=tf_s), batch_size=64)
    correct = n = 0
    with torch.no_grad():
        for x, y in dl:
            correct += (r50(x.to(device))[:, subset.to(device)].argmax(1).cpu() == y).sum().item(); n += len(y)
    print(f"{size:>3}px: acc {correct / n:.4f}")
# Too small loses detail. Larger than training can help a little (objects look the size the model
# learned), until it starts hurting. Imagenette-160's source images are only 160 px, so 288 is upsampled.

**Experiment 3: batch size vs. throughput.** Predict: does MobileNetV3 (5× fewer FLOPs) stay 5× faster than ResNet-50 as the batch grows?

In [ ]:
# Prediction: ...
print(f"{'batch':>6}{'MobileNetV3 img/s':>20}{'ResNet-50 img/s':>18}{'ratio':>8}")
mob = timm.create_model("mobilenetv3_large_100", pretrained=False).eval().to(device)
res = timm.create_model("resnet50", pretrained=False).eval().to(device)
for bsz in (1, 8, 32, 64):
    a = bsz / bench(mob, 224, bsz, reps=5); b = bsz / bench(res, 224, bsz, reps=5)
    print(f"{bsz:>6}{a:>20.0f}{b:>18.0f}{a / b:>8.1f}")
# MobileNetV3 has ~19× fewer FLOPs than ResNet-50, yet it is only ~2× faster at batch 1 and at most ~6× at larger batches.
# Small batches: per-layer launch overhead dominates, and MobileNet's many cheap layers can't keep the GPU busy.
# Larger batches give each launch more work, so the gap grows, but it never approaches 19×. FLOPs ≠ speed, again.

---
# 📝 8 · Recap

**Cheat sheet**
- **VGG**: stack 3×3s. **Inception**: parallel branches + 1×1 bottlenecks. **ResNet**: $y = x + F(x)$. **MobileNet**: depthwise-separable + inverted residuals. **SE**: channel re-weighting. **EfficientNet**: compound scaling. **ConvNeXt**: ResNet modernized with transformer-era tricks.
- Skip connections give gradients a highway ($1 + \partial F / \partial x$) and make very deep nets trainable.
- Bottlenecks put the expensive 3×3 on fewer channels: ~8× cheaper than one plain 3×3 at 256 channels (~17× vs. two).
- `timm.create_model(name, pretrained=True)` + **its own data config** (size, interpolation, mean/std).
- Compare models on **accuracy, params, FLOPs and measured latency/throughput on your hardware**. FLOPs ≠ speed.
- **Grad-CAM**: gradient-weighted feature maps show where the model looked. Use it to debug mistakes and spot shortcuts.

**Interview-style questions**
1. What problem did ResNet solve? Why can't a deeper plain network just learn identity layers?
2. Derive the gradient through a residual block. Why does it help?
3. Compute the parameter savings of a ResNet bottleneck vs. two 3×3 convs at 256 channels.
4. What is an inverted residual, and why is its last 1×1 conv linear (no ReLU)?
5. What does a squeeze-and-excitation block do, and what does it cost?
6. Why can a model with fewer FLOPs be slower? Give two reasons.
7. How does Grad-CAM work? What does a heatmap on the background suggest?

**Next:** `05_transfer_learning`: adapting these pretrained models to *your* data.